Acknowledgement: Consulted with Claude for the framework of code. AI-generated code in this project is manually debugged and heavily modified. As a result, the majority of the code in this file is manually written.

In [1]:
import sqlite3
import csv

def create_flights_table(conn):
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS flights (
        flight_id TEXT PRIMARY KEY,
        source TEXT,
        destination TEXT,
        capacity INTEGER,
        seats_remaining INTEGER
    );
    """
    conn.execute(create_table_sql)
    conn.commit()

def load_csv_to_table(conn, csv_path, table_name="flights"):
    cursor = conn.cursor()
    cursor.execute(f"DELETE FROM {table_name}")  # clear existing rows first

    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        rows = [
            (row["flight_id"], row["source"], row["destination"], row["capacity"], row["seats_remaining"])
            for row in reader
        ]

    sql = f"""
        INSERT INTO {table_name} (flight_id, source, destination, capacity, seats_remaining)
        VALUES (?, ?, ?, ?, ?)
    """
    cursor.executemany(sql, rows)
    conn.commit()
    print(f"Loaded {len(rows)} rows into {table_name} table.")

if __name__ == "__main__":
    conn = sqlite3.connect("flights_database.db") # NOTE: Claude suggested saving to the database file (.db), so the data persists after the notebook session ends
    create_flights_table(conn)          # create the table first
    load_csv_to_table(conn, "flights.csv")
    conn.close()

Loaded 200 rows into flights table.


In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flights_database.db")

def query(sql):
    """Helper: run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn)

# 1. View all flights
print(query("SELECT * FROM flights"))

# 2. Flights with available seats
print(query("""
    SELECT flight_id, source, destination, seats_remaining
    FROM flights
    WHERE seats_remaining > 0
"""))

# 3. Fully booked flights
print(query("""
    SELECT flight_id, source, destination, seats_remaining
    FROM flights
    WHERE seats_remaining = 0
"""))


    flight_id     source destination  capacity  seats_remaining
0         PA0    Calgary   Vancouver       150                6
1         PA1    Toronto      Ottawa       175               35
2         PA2    Toronto      Ottawa       250              108
3         PA3  Vancouver     Toronto       150               55
4         PA4     Ottawa    Montreal       250               50
..        ...        ...         ...       ...              ...
195     PA195    Calgary   Vancouver       150              141
196     PA196    Toronto     Calgary       225               26
197     PA197   Montreal     Calgary       225              122
198     PA198    Calgary    Montreal       175               13
199     PA199   Montreal      Ottawa       150               16

[200 rows x 5 columns]
    flight_id     source destination  seats_remaining
0         PA0    Calgary   Vancouver                6
1         PA1    Toronto      Ottawa               35
2         PA2    Toronto      Ottawa          

For the purpose of proving the concept of our table schema, I will use some mock data directly assigned to variables

In [3]:
import sqlite3
import csv
import uuid

In [9]:
def create_flights_table(conn):
    conn.execute("DROP TABLE IF EXISTS flights")  # remove old schema
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS flights (
        flight_uuid TEXT PRIMARY KEY,
        flight_number TEXT,
        origin TEXT,
        destination TEXT,
        capacity INTEGER,
        seats_remaining INTEGER
    );
    """
    conn.execute(create_table_sql)
    conn.commit()


In [10]:
def load_csv_to_table(conn, csv_path, table_name="flights"):
    cursor = conn.cursor()
    cursor.execute(f"DELETE FROM {table_name}")

    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        rows = [
            (
                str(uuid.uuid4()),          # generate a fresh UUID per row
                row["flight_number"],
                row["origin"],
                row["destination"],
                row["capacity"],
                row["seats_remaining"],
            )
            for row in reader
        ]

    sql = f"""
        INSERT INTO {table_name} (flight_uuid, flight_number, origin, destination, capacity, seats_remaining)
        VALUES (?, ?, ?, ?, ?, ?)
    """
    cursor.executemany(sql, rows)
    conn.commit()
    print(f"Loaded {len(rows)} rows into {table_name} table.")


In [14]:
if __name__ == "__main__":
    conn = sqlite3.connect("flights_database.db")
    try:
        create_flights_table(conn)
        load_csv_to_table(conn, "flights.csv")
    finally:
        print("\"flights\" table created and loaded successfully.")
        # conn.close()


Loaded 200 rows into flights table.
"flights" table created and loaded successfully.


In [15]:
def query(sql):
    """Helper: run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn)


In [16]:
# 1. View all flights
print(query("SELECT * FROM flights"))


                              flight_uuid flight_number     origin  \
0    c5df8811-f82b-44d1-a35b-37ee3ba3dda0           PA0    Calgary   
1    ccf53854-975a-49a0-a0c3-4b5634c4941b           PA1    Toronto   
2    975674c1-db38-4de6-a541-4172ed7381cb           PA2    Toronto   
3    e1217039-00eb-4788-afed-ed1a5315d257           PA3  Vancouver   
4    d21c05c6-3a7c-4697-aa7d-0c2347823f74           PA4     Ottawa   
..                                    ...           ...        ...   
195  de139735-c407-490e-aced-59356fa14c5f         PA195    Calgary   
196  cf093987-4395-42c4-b489-9be258a7a59b         PA196    Toronto   
197  562cae02-e4d3-4582-a4eb-8ef6ffa813a7         PA197   Montreal   
198  e4765ef6-2f96-4e99-849c-3d46e1b1b6af         PA198    Calgary   
199  5fe9b010-9c98-4d79-9269-b892cd17d02d         PA199   Montreal   

    destination  capacity  seats_remaining  
0     Vancouver       150                6  
1        Ottawa       175               35  
2        Ottawa       25

In [18]:
# 2. Flights with available seats
print(query("""
    SELECT flight_uuid, flight_number, origin, destination, seats_remaining
    FROM flights
    WHERE seats_remaining > 0
"""))


                              flight_uuid flight_number     origin  \
0    c5df8811-f82b-44d1-a35b-37ee3ba3dda0           PA0    Calgary   
1    ccf53854-975a-49a0-a0c3-4b5634c4941b           PA1    Toronto   
2    975674c1-db38-4de6-a541-4172ed7381cb           PA2    Toronto   
3    e1217039-00eb-4788-afed-ed1a5315d257           PA3  Vancouver   
4    d21c05c6-3a7c-4697-aa7d-0c2347823f74           PA4     Ottawa   
..                                    ...           ...        ...   
193  de139735-c407-490e-aced-59356fa14c5f         PA195    Calgary   
194  cf093987-4395-42c4-b489-9be258a7a59b         PA196    Toronto   
195  562cae02-e4d3-4582-a4eb-8ef6ffa813a7         PA197   Montreal   
196  e4765ef6-2f96-4e99-849c-3d46e1b1b6af         PA198    Calgary   
197  5fe9b010-9c98-4d79-9269-b892cd17d02d         PA199   Montreal   

    destination  seats_remaining  
0     Vancouver                6  
1        Ottawa               35  
2        Ottawa              108  
3       Toronto    

In [ ]:
# 3. Fully booked flights
print(query("""
    SELECT flight_id, source, destination, seats_remaining
    FROM flights
    WHERE seats_remaining = 0
"""))
